# Super-resolved Sentinel-2 on demand

This notebook sends an area and a date to the SR service and gets back 2.5 m imagery made
from 10 m Sentinel-2. It takes about 10 minutes, most of which is waiting for the GPUs.

1. Install the client and enter your API key.
2. Estimate the cost of an area and see which Sentinel-2 scenes are available.
3. Run stage 1, which is fast and multi-image, then add stage 2, the diffusion refinement.
4. Look at the result, open it as an xarray, and download the files.

In [ ]:
%pip install -q "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"

## 1. Connect

Your API key starts with `srk_`. In Colab you can store it under **Secrets** (the key icon
on the left) as `SR_API_KEY`; otherwise the cell asks for it. The service URL is given at the
workshop.

In [ ]:
import getpass
import os

def secret(name, prompt):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return getpass.getpass(prompt) if "KEY" in name else input(prompt)

os.environ["SR_API_KEY"] = secret("SR_API_KEY", "API key (srk_...): ")
os.environ["SR_API_URL"] = secret("SR_API_URL", "Service URL: ")

from sr_client import SRClient, area_around

sr = SRClient()
sr.me()

## 2. Pick an area and estimate

`area_around(lat, lon, km)` makes a square. You can also pass a bounding box
`[W, S, E, N]` or any GeoJSON polygon. Estimates are free: they show the tile count, the cost
(assuming nothing is cached yet), the Sentinel-2 scenes in the time window, and warnings that
are likely to fire.

In [ ]:
area = area_around(14.5995, 120.9842, km=2)   # Manila; try your own place
target_date = "2025-12-01"

est = sr.estimate(area, target_date)
print(f"{est['tiles']} tiles, about ${est['estimated_usd']:.2f}")
for s in est["scenes"]:
    print(f"  {s['date']}  cloud {s['cloud_cover']:5.1f} %  {s['id']}")
for w in est["warnings_likely"]:
    print("warning:", w["message"])
for p in est["problems"]:
    print("cannot submit:", p)

## 3. Run it

Stage 1 alone takes a minute or two and costs very little. `job.upgrade()` then adds stage 2,
the diffusion refiner, which reuses stage 1's tiles. You can also submit both stages at once,
which is the default.

In [ ]:
job = sr.submit(area, target_date, stages=["1"])
job.wait()
job.show()

In [ ]:
job.upgrade().wait()
job.show()

## 4. Use the result

`job.open()` reads a GeoTIFF straight from its download link as an xarray `DataArray` in
reflectance units. Below, the 10 m input and both stages are compared over a zoomed window.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lr = job.open("lr_reference.tif")
s1 = job.open("stage1_sr.tif")
s2 = job.open("stage2_sr.tif")

def rgb(da, y0, x0, size):
    a = da.isel(band=[0, 1, 2], y=slice(y0, y0 + size), x=slice(x0, x0 + size)).values
    return np.clip(np.nan_to_num(np.moveaxis(a, 0, -1)) / 0.3, 0, 1)

h, w = s2.sizes["y"], s2.sizes["x"]
y0, x0, size = h // 2 - 100, w // 2 - 100, 200          # a 500 m window at 2.5 m
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, [("Sentinel-2, 10 m", rgb(lr, y0 // 4, x0 // 4, size // 4)),
                                   ("stage 1, 2.5 m", rgb(s1, y0, x0, size)),
                                   ("stage 2, 2.5 m", rgb(s2, y0, x0, size))]):
    ax.imshow(img, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()

ndvi = (s2.sel(band=4) - s2.sel(band=1)) / (s2.sel(band=4) + s2.sel(band=1))
ndvi.plot(cmap="RdYlGn", vmin=-0.2, vmax=0.9, figsize=(7, 6))
plt.title("NDVI from the stage-2 output")
plt.show()

In [ ]:
for w in job.warnings:
    print(f"[{w['code']}] {w['message']}")

paths = job.download("sr_results")
paths

In Colab, the files are in the file browser on the left, under `sr_results/<job id>/`.
`manifest.json` lists the scenes used for every tile, the model versions and the cost.
`sr.me()` shows how much budget your key has left, and `sr.jobs()` lists your jobs.

Results contain modified Copernicus Sentinel data.

In [ ]:
sr.me()["budget"]